# Modeling Population Validation

This notebook validates the proposed modeling design before abuse ground truth is introduced.

Goals:
- Convert the raw online and offline records into minimal atomic events
- Evaluate candidate prediction dates
- Measure how much historical activity users have
- Determine whether the proposed eligibility rules retain enough users
- Finalize the modeling population before constructing abuse cases

In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")

offline = pd.read_csv(
    RAW_DIR / "ccf_offline_stage1_train.csv",
    usecols=["User_id", "Coupon_id", "Date_received", "Date"]
)

online = pd.read_csv(
    RAW_DIR / "ccf_online_stage1_train.csv",
    usecols=["User_id", "Action", "Coupon_id", "Date_received", "Date"]
)

## 1. Construct Minimal Offline Events

An offline source row may describe more than one event.

For example, a coupon can be received on one date and redeemed on another.
For population analysis, these are represented as separate atomic events.

In [2]:
offline_receipts = offline.loc[
    offline["Date_received"].notna(),
    ["User_id", "Date_received"]
].copy()

offline_receipts["event_date"] = pd.to_datetime(
    offline_receipts["Date_received"].astype("Int64").astype(str),
    format="%Y%m%d",
    errors="coerce"
)

offline_receipts["event_type"] = "coupon_received"
offline_receipts["is_coupon_receipt"] = 1

offline_purchases = offline.loc[
    offline["Date"].notna(),
    ["User_id", "Coupon_id", "Date"]
].copy()

offline_purchases["event_date"] = pd.to_datetime(
    offline_purchases["Date"].astype("Int64").astype(str),
    format="%Y%m%d",
    errors="coerce"
)

offline_purchases["event_type"] = "purchase"
offline_purchases["is_coupon_receipt"] = 0

offline_events = pd.concat(
    [
        offline_receipts[
            ["User_id", "event_date", "event_type", "is_coupon_receipt"]
        ],
        offline_purchases[
            ["User_id", "event_date", "event_type", "is_coupon_receipt"]
        ],
    ],
    ignore_index=True
)

offline_events["channel"] = "offline"

## 2. Construct Minimal Online Events

The online source already contains an Action field.

Action:
- 0 = click
- 1 = purchase
- 2 = coupon receipt

In [3]:
online_events = online.copy()

online_events["event_date"] = pd.NaT

online_events.loc[
    online_events["Action"].isin([0, 1]),
    "event_date"
] = pd.to_datetime(
    online_events.loc[
        online_events["Action"].isin([0, 1]),
        "Date"
    ].astype("Int64").astype(str),
    format="%Y%m%d",
    errors="coerce"
)

online_events.loc[
    online_events["Action"] == 2,
    "event_date"
] = pd.to_datetime(
    online_events.loc[
        online_events["Action"] == 2,
        "Date_received"
    ].astype("Int64").astype(str),
    format="%Y%m%d",
    errors="coerce"
)

action_map = {
    0: "click",
    1: "purchase",
    2: "coupon_received"
}

online_events["event_type"] = online_events["Action"].map(action_map)

online_events["is_coupon_receipt"] = (
    online_events["Action"] == 2
).astype(int)

online_events["channel"] = "online"

online_events = online_events[
    [
        "User_id",
        "event_date",
        "event_type",
        "is_coupon_receipt",
        "channel",
    ]
]

all_events = pd.concat(
    [
        offline_events[
            ["User_id", "event_date", "event_type", "is_coupon_receipt", "channel"]
        ],
        online_events
    ],
    ignore_index=True
)

all_events = all_events.dropna(subset=["event_date"])

user_daily = (
    all_events
    .groupby(["User_id", "event_date"], as_index=False)
    .agg(
        event_count=("event_type", "size"),
        coupon_receipts=("is_coupon_receipt", "sum")
    )
)

user_daily.head()

,User_id,event_date,event_count,coupon_receipts
0,4,2016-02-14,1,1
1,4,2016-03-08,1,0
2,4,2016-03-31,1,1
3,4,2016-06-07,1,1
4,35,2016-01-29,2,2


In [4]:
prediction_dates = pd.date_range(
    start="2016-02-01",
    end="2016-06-06",
    freq="7D"
)

prediction_dates

DatetimeIndex(['2016-02-01', '2016-02-08', '2016-02-15', '2016-02-22',
               '2016-02-29', '2016-03-07', '2016-03-14', '2016-03-21',
               '2016-03-28', '2016-04-04', '2016-04-11', '2016-04-18',
               '2016-04-25', '2016-05-02', '2016-05-09', '2016-05-16',
               '2016-05-23', '2016-05-30', '2016-06-06'],
              dtype='datetime64[us]', freq='7D')

In [5]:
results = []

for prediction_date in prediction_dates:

    history_start = prediction_date - pd.Timedelta(days=30)

    history = user_daily[
        (user_daily["event_date"] >= history_start)
        & (user_daily["event_date"] < prediction_date)
    ]

    user_history = (
        history
        .groupby("User_id", as_index=False)
        .agg(
            historical_events=("event_count", "sum"),
            historical_coupon_receipts=("coupon_receipts", "sum")
        )
    )

    results.append({
        "prediction_date": prediction_date,
        "users_with_1_event":
            (user_history["historical_events"] >= 1).sum(),

        "users_with_5_events":
            (user_history["historical_events"] >= 5).sum(),

        "users_with_10_events":
            (user_history["historical_events"] >= 10).sum(),

        "users_with_coupon":
            (user_history["historical_coupon_receipts"] >= 1).sum(),

        "eligible_5_events_coupon":
            (
                (user_history["historical_events"] >= 5)
                & (user_history["historical_coupon_receipts"] >= 1)
            ).sum(),

        "eligible_10_events_coupon":
            (
                (user_history["historical_events"] >= 10)
                & (user_history["historical_coupon_receipts"] >= 1)
            ).sum()
    })

population_summary = pd.DataFrame(results)

display(population_summary)

population_summary.describe()

,prediction_date,users_with_1_event,users_with_5_events,users_with_10_events,users_with_coupon,eligible_5_events_coupon,eligible_10_events_coupon
0,2016-02-01,503941,132410,52761,215005,48869,16085
1,2016-02-08,525464,121027,45459,268965,51128,15899
2,2016-02-15,501017,101687,35757,276279,46395,13194
3,2016-02-22,478366,89904,29702,269469,42358,11159
4,2016-02-29,405129,70416,26118,180036,21051,6891
5,2016-03-07,380784,84541,33934,111025,20890,8453
6,2016-03-14,403110,105925,44415,92536,25603,11169
7,2016-03-21,412988,112683,47669,89081,26533,11765
8,2016-03-28,446855,123974,51189,132347,37861,14643
9,2016-04-04,457301,125250,51197,144218,38729,14584


,prediction_date,users_with_1_event,users_with_5_events,users_with_10_events,users_with_coupon,eligible_5_events_coupon,eligible_10_events_coupon
count,19,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,2016-04-04 00:00:00,473850.526316,115539.842105,46316.473684,180579.052632,37234.105263,13349.473684
min,2016-02-01 00:00:00,380784.000000,70416.000000,26118.000000,89081.000000,20890.000000,6891.000000
25%,2016-03-03 12:00:00,452078.000000,109304.000000,44937.000000,151679.500000,32780.500000,12283.000000
50%,2016-04-04 00:00:00,492035.000000,121889.000000,50177.000000,179800.000000,38050.000000,13194.000000
75%,2016-05-05 12:00:00,502479.000000,126267.500000,51283.500000,212117.500000,43727.500000,14914.500000
max,2016-06-06 00:00:00,525464.000000,132663.000000,53756.000000,276279.000000,51128.000000,17416.000000
std,NaN,44142.048156,17379.861911,8456.680931,54967.766297,9111.092116,2655.846765
